In [ ]:
# Load necessary libraries
install.packages(c("dplyr", "tidyr"))


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
library(dplyr)
library(tidyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
# Read the data; make sure the name in the file upload matches the name in the read_csv bubble
# Do not change the skeleton
data <- read.csv("Data.csv")
print(data)

    ID.Number Gender Age       Date Moon.Diameter..km. Moon.Distance..km.
1        1000 Female  19  5/22/2024               3476             395236
2        1001   Male  24  5/22/2024               3476             395236
3        1002   Male  19  5/22/2024               3476             395236
4        1003 Female  19  5/22/2024               3476             395236
5        1004 Female  18  5/22/2024               3476             395236
6        1005   Male  19  5/22/2024               3476             395236
7        1006 Female  18  5/22/2024               3476             395236
8        1007 Female  18  5/22/2024               3476             395236
9        1008 Female  19  5/22/2024               3476             395236
10       1009   Male  18  5/22/2024               3476             395236
11       1010 Female  18  5/22/2024               3476             395236
12       1011 Female  21  5/22/2024               3476             395236
13       1012   Male  17  7/20/2024   

In [ ]:
# Function to calculate visual angle
calc_va_df <- function(df, size_col, dist_col) {
  size <- df[[size_col]]
  dist <- df[[dist_col]]
  (2 * atan(size / (2 * dist))) * (180 / pi)
}

# Calculate Visual Angle of All Objects
data <- data %>%
  mutate(
  # Visual Angles produced by probes
    Lower_Elevation_Perceptual_VA  = calc_va_df(data, "Round.1.Estimate.Template.Size..cm.", "Round.1.Estimate.Distance..cm."),
    Lower_Elevation_Adjusted_VA  = calc_va_df(data, "Round.1.Adjusted.Template.Size..cm.", "Round.1.Adjusted.Distance..cm."),
    Higher_Elevation_Perceptual_VA  = calc_va_df(data, "Round.2.Estimate.Template.Size..cm.", "Round.2.Estimate.Distance..cm."),
    Higher_Elevation_Adjusted_VA  = calc_va_df(data, "Round.2.Adjusted.Template.Size..cm.", "Round.2.Adjusted.Distance..cm."),
# Visual Angle of the Moon
    Real_Visual_Angle = calc_va_df(data, "Moon.Diameter..km.", "Moon.Distance..km."),
# Disparity Visual Angle
    Lower_Elevation_Disparity = calc_va_df(data, "Round.1.Caliper.Aperture..cm.", "Round.1.Caliper.Distance..cm."),
    Higher_Elevation_Disparity = calc_va_df(data, "Round.2.Caliper.Aperture..cm.", "Round.2.Caliper.Distance..cm."),
    Diameter = data$Moon.Diameter..km.,
    Distance = data$Moon.Distance..km.,
    Lower_Elevation = data$Elevation..deg.,
    Higher_Elevation = data$`Elevation..deg..1`)

data_long <- data %>%
  select(ID = 1, Gender, Date, Time, Age, Distance,
        Lower_Elevation_Perceptual_VA, Lower_Elevation_Adjusted_VA,
        Higher_Elevation_Perceptual_VA, Higher_Elevation_Adjusted_VA, Real_Visual_Angle,
        Lower_Elevation, Higher_Elevation, Lower_Elevation_Disparity, Higher_Elevation_Disparity) %>%
  pivot_longer(
    cols = c(Lower_Elevation_Perceptual_VA, Lower_Elevation_Adjusted_VA,
        Higher_Elevation_Perceptual_VA, Higher_Elevation_Adjusted_VA,),
    names_to = "Measurement",
    values_to = "Reported_Visual_Angle") %>%
  filter(!is.na(Reported_Visual_Angle)) %>%
  mutate(
    Task = case_when(
      grepl("Perceptual", Measurement) ~ "Perceptual",
      grepl("Adjusted", Measurement) ~ "Adjusted"
    ),
    Ratio_Visual_Angle = Reported_Visual_Angle / Real_Visual_Angle,

    Elevation = case_when(
      grepl("Lower_Elevation", Measurement) ~ Lower_Elevation,
      grepl("Higher_Elevation", Measurement) ~ Higher_Elevation
    ),
    Session = case_when(
      grepl("Lower", Measurement) ~ "Lower",
      grepl("Higher", Measurement) ~ "Higher"
    ),
    # Calculate Disparity_VA using the pre-calculated columns
    Disparity_VA = case_when(
      Session == "Lower" ~ Lower_Elevation_Disparity,
      Session == "Higher" ~ Higher_Elevation_Disparity
    )
  )

data_long <- data_long %>% select(-Higher_Elevation, -Lower_Elevation, -Lower_Elevation_Disparity, -Higher_Elevation_Disparity, -Measurement)
data_long <- filter(data_long)
write.csv(data_long, file = "FullMoonDataLong.csv", row.names = FALSE)